In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Chuẩn bị dữ liệu

In [7]:
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [12]:
data_dir = "/kaggle/input/datasets/grassknoted/asl-alphabet/asl_alphabet_train/asl_alphabet_train"
input_size = (128, 128) # Các ảnh sẽ được resize về một kích thước chung, độ phân giải càng cao -> càng tốn tài nguyên
val_frac = 0.1 # Tỉ lệ của tập validation sẽ được chia ra từ tập Train => 9:1, 8:2, 7:3
batch_size = 128 # Số ảnh xử lý/ lần để cập nhật tham số, số càng cao càng tốn tài nguyên

## Tiền xử lý dữ liệu ảnh để mô hình học tốt hơn. 
data_augmentor = ImageDataGenerator(samplewise_center=True, 
                                    samplewise_std_normalization=True,
                                    validation_split=val_frac)

# Đọc dữ liệu từ thư mục và tiền xử lý
train_generator = data_augmentor.flow_from_directory(data_dir,
                                                     target_size=input_size,
                                                     batch_size=batch_size,
                                                     shuffle=True, 
                                                     subset="training")


val_generator = data_augmentor.flow_from_directory(data_dir,
                                                     target_size=input_size,
                                                     batch_size=batch_size,
                                                     subset="validation")


Found 78300 images belonging to 29 classes.
Found 8700 images belonging to 29 classes.


In [13]:
# Có 29 lớp trong tập train, tương ứng với 29 folders
train_generator.class_indices

{'A': 0,
 'B': 1,
 'C': 2,
 'D': 3,
 'E': 4,
 'F': 5,
 'G': 6,
 'H': 7,
 'I': 8,
 'J': 9,
 'K': 10,
 'L': 11,
 'M': 12,
 'N': 13,
 'O': 14,
 'P': 15,
 'Q': 16,
 'R': 17,
 'S': 18,
 'T': 19,
 'U': 20,
 'V': 21,
 'W': 22,
 'X': 23,
 'Y': 24,
 'Z': 25,
 'del': 26,
 'nothing': 27,
 'space': 28}

# Xây dựng mô hình

In [15]:
num_classes = len(train_generator.class_indices) # 29
input_shape = (128, 128, 3) # 3 chiều cho ảnh RGB

# Cấu trúc mô hình tham khảo
model = keras.models.Sequential([
    keras.Input(shape=input_shape),
    # Block 1
    keras.layers.Conv2D(32, kernel_size=(3,3), activation="relu"), # layer tích chập
    keras.layers.Conv2D(32, kernel_size=(3,3), activation="relu"), # thêm layer tích chập để học nhiều chi tiết hơn.
    keras.layers.MaxPooling2D(pool_size=(2,2)), # MaxPool luôn theo sau layer Conv
    keras.layers.Dropout(0.5), # Để mô hình không bị thiên kiến
    

    # Block 2
    keras.layers.Conv2D(64, kernel_size=(3,3), activation="relu"), # layer tích chập
    keras.layers.Conv2D(64, kernel_size=(3,3), activation="relu"), # thêm layer tích chập để học nhiều chi tiết hơn.
    keras.layers.MaxPooling2D(pool_size=(2,2)), # MaxPool luôn theo sau layer Conv
    keras.layers.Dropout(0.5), # Để mô hình không bị thiên kiến

    # Block 3
    keras.layers.Conv2D(128, kernel_size=(3,3), activation="relu"), # layer tích chập
    keras.layers.Conv2D(128, kernel_size=(3,3), activation="relu"), # thêm layer tích chập để học nhiều chi tiết hơn.
    keras.layers.MaxPooling2D(pool_size=(2,2)), # MaxPool luôn theo sau layer Conv
    keras.layers.Dropout(0.5), # Để mô hình không bị thiên kiến

    # FCN block chịu trách nhiệm phân loại
    keras.layers.Flatten(), # Chỉ một layer Flatten trong mô hình.


    keras.layers.Dense(512, activation="relu"),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(num_classes, activation="softmax"),
    # Layer Dense cuối có chiều là tổng số class
        
])


model.compile(optimizer="adam", # luôn sử dụng "adam"
              loss="categorical_crossentropy", 
              # binary_crossentropy cho lớp True/False
              # categorical_crossentropy cho nhiều lớp (folder nhãn)
              metrics=["accuracy"]
             )
model.summary()

2026-04-05 08:59:31.103030: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 124, 124, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 62, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 62, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 60, 60, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 58, 58, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 29, 29, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 29, 29, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 27, 27, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 25, 25, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     9,437,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 29)             │         3,741 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,794,109 (37.36 MB)

 Trainable params: 9,794,109 (37.36 MB)

 Non-trainable params: 0 (0.00 B)

# Huấn luyện mô hình

In [16]:
model.fit(train_generator,
          epochs=5,
          validation_data=val_generator
         )

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
 16/612 ━━━━━━━━━━━━━━━━━━━━ 1:00:46 6s/step - accuracy: 0.0366 - loss: 3.4292

KeyboardInterrupt: 